# 🐍 Python from Scratch — Module 5: Error Handling and Files

### When a program meets reality

Data from a user, a file that doesn't exist, dividing by zero — the world is rarely as
clean as the examples in earlier modules. This notebook covers how to make a program
react to problems in a controlled way instead of crashing, and how to permanently save
and load data from disk.

## Table of Contents

1. [Anatomy of an error — the traceback](#sec1)
2. [`try` / `except` — the basics](#sec2)
3. [Catching specific exceptions](#sec3)
4. [`else` and `finally`](#sec4)
5. [Raising your own errors — `raise`](#sec5)
6. [Reading files](#sec6)
7. [Writing files](#sec7)
8. [Fun fact: EAFP vs LBYL](#sec8)
9. [Module summary](#sec9)
10. [Exercises](#sec10)

---

<a id="sec1"></a>
## 1. Anatomy of an error — the traceback

When something goes wrong in Python, the program stops and prints a **traceback** —
information about which line the error happened on and why. This isn't something to
fear — it's the most useful clue for fixing your code. Read a traceback **from the
bottom up**: the last line tells you the error type and exactly what went wrong.

In [ ]:
# Run this on purpose, to see a traceback:
number = int("not a number")

> 💡 **How to read a traceback**
>
> The last line (`ValueError: invalid literal for int() with base 10: 'not a number'`) tells you exactly what went wrong. The lines above show the call path that led there - in simple scripts, usually the last line plus the one pointing at a line number in your own code is all you need to look at.

<a id="sec2"></a>
## 2. `try` / `except` — the basics

A `try` block "attempts" to run some code, and if an error (exception) happens inside
it, control jumps to the `except` block instead of crashing the whole program.

In [ ]:
try:
    number = int("not a number")
    print("This line never runs")
except ValueError:
    print("Couldn't convert the text to a number")

print("The program keeps running normally")

<a id="sec3"></a>
## 3. Catching specific exceptions

It's worth catching a **specific** exception type rather than everything at once (a bare
`except:`) — that way you don't accidentally hide completely different, unexpected
errors. You can handle several types in one block, or with separate `except` blocks.

In [ ]:
def divide(a, b):
    try:
        result = a / b
        return result
    except ZeroDivisionError:
        print("You can't divide by zero!")
        return None
    except TypeError:
        print("Both arguments must be numbers!")
        return None

print(divide(10, 2))
print(divide(10, 0))
print(divide(10, "abc"))

# Several types in one except:
try:
    my_dict = {"a": 1}
    print(my_dict["b"])
except (KeyError, IndexError) as error:
    print(f"Something wasn't found: {error}")

> ⚠️ **Avoid a bare `except:`**
>
> `except:` with no type catches **literally everything** - including a typo in a variable name or an attempt to interrupt the program with Ctrl+C. That makes real bugs in your code much harder to spot, since everything gets silently swallowed. Always name the specific exception type you expect.

<a id="sec4"></a>
## 4. `else` and `finally`

The full `try` construct has two more optional blocks:

- **`else`** — runs only if `try` finished **without** an error,
- **`finally`** — runs **always**, whether there was an error or not (handy for e.g.
  cleaning up resources).

In [ ]:
def divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Error: division by zero")
    else:
        print(f"Result: {result}")   # only when it succeeded
    finally:
        print("Division attempt finished")   # always

divide(10, 2)
print("---")
divide(10, 0)

<a id="sec5"></a>
## 5. Raising your own errors — `raise`

Sometimes your own code should raise an error when data breaks some business rule (e.g.
age can't be negative). You do this with the `raise` statement, along with the
appropriate exception type and message.

In [ ]:
def set_age(age):
    if age < 0:
        raise ValueError("Age can't be negative")
    return age

print(set_age(30))

try:
    set_age(-5)
except ValueError as error:
    print(f"Validation error: {error}")

<a id="sec6"></a>
## 6. Reading files

A file is opened with the `open()` function, best used inside a `with` block — that way
the file gets **closed automatically**, even if an error happens inside. Without `with`,
you'd have to remember to call `.close()` manually.

In [ ]:
# First, let's create a file to read (writing is covered properly in the next section):
with open("example.txt", "w", encoding="utf-8") as file:
    file.write("First line\n")
    file.write("Second line\n")
    file.write("Third line\n")

# Reading everything at once:
with open("example.txt", "r", encoding="utf-8") as file:
    contents = file.read()
print(contents)

# Reading line by line (handy for large files):
with open("example.txt", "r", encoding="utf-8") as file:
    for line in file:
        print("Line:", line.strip())   # .strip() removes the trailing newline

> ⚠️ **A file that doesn't exist**
>
> Trying to open a file that doesn't exist in `"r"` (read) mode raises a `FileNotFoundError`. That's a good candidate for `try`/`except`, especially when the file name comes from the user.

<a id="sec7"></a>
## 7. Writing files

The mode you open a file in decides what happens to its contents:

| Mode | Meaning |
|---|---|
| `"w"` | write — **overwrites** the whole file (or creates a new one) |
| `"a"` | append — adds to the end, without erasing existing content |
| `"r"` | read (the default, if you don't specify a mode) |

In [ ]:
# "w" overwrites the whole file:
with open("log.txt", "w", encoding="utf-8") as file:
    file.write("Program start\n")

# "a" appends, without erasing earlier content:
with open("log.txt", "a", encoding="utf-8") as file:
    file.write("Event 1\n")

with open("log.txt", "a", encoding="utf-8") as file:
    file.write("Event 2\n")

with open("log.txt", "r", encoding="utf-8") as file:
    print(file.read())

<a id="sec8"></a>
## 8. Fun fact: EAFP vs LBYL

Two philosophies for handling potential errors.

In [ ]:
my_dict = {"a": 1}

# LBYL - "Look Before You Leap" (check first, then try):
if "b" in my_dict:
    print(my_dict["b"])
else:
    print("No 'b' key")

# EAFP - "Easier to Ask Forgiveness than Permission" (try it, handle the error if it happens):
try:
    print(my_dict["b"])
except KeyError:
    print("No 'b' key")

> 💡 **Fun fact**
>
> Python has traditionally favored the EAFP style - `try`/`except` is often faster and considered more «Pythonic» than a series of upfront `if` checks, especially when the error is a rare edge case rather than the normal path through the program.

<a id="sec9"></a>
## 9. Module summary

By now it should be clear:

- how to read a traceback to understand what went wrong,
- how `try`/`except` works, and why it's worth catching specific exception types,
- what `else` and `finally` are for inside a `try` block,
- how to raise your own errors with `raise`,
- how to safely read and write files with `with open(...)`,
- the difference between `"r"`, `"w"`, and `"a"` modes.

This wraps up the core Python course that started back in Module 1. The natural next
step from here is more advanced territory: object-oriented programming (classes),
working with modules and packages, or specific libraries (e.g. for data analysis) —
depending on which direction you want to take next.

<a id="sec10"></a>
## 10. Exercises

The final set of exercises in this series — several combine error handling with files,
to practice both topics at once.

> 📝 **Exercise 1: Safe division**
>
> Write a function `safe_divide(a, b)` that returns the result of `a / b`, and if `b` is 0 - catches `ZeroDivisionError`, prints a message, and returns `None`.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Can't divide by zero")
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))
```
</details>

> 📝 **Exercise 2: Validating age input in a loop**
>
> Write a `while True` loop that asks the user for their age via `input()`, tries to convert it to `int`, and if that fails (`ValueError`) - prints a message and asks again. It ends once the user enters a valid number.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
while True:
    text = input("Enter your age: ")
    try:
        age = int(text)
        print(f"Thanks, you are {age} years old")
        break
    except ValueError:
        print("That's not a valid number, try again")
```
</details>

> 📝 **Exercise 3: Calculator with multiple error handling**
>
> Write a function `calculator(a, b, operation)`, where `operation` is a string `"+"`, `"-"`, `"*"`, or `"/"`. Handle `ZeroDivisionError` for division by zero, and the case where `operation` isn't one of the known ones (return a message about an unknown operation instead of an error).

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
def calculator(a, b, operation):
    if operation == "+":
        return a + b
    elif operation == "-":
        return a - b
    elif operation == "*":
        return a * b
    elif operation == "/":
        try:
            return a / b
        except ZeroDivisionError:
            return "Error: division by zero"
    else:
        return f"Unknown operation: {operation}"

print(calculator(10, 2, "/"))
print(calculator(10, 0, "/"))
print(calculator(10, 2, "%"))
```
</details>

> 📝 **Exercise 4: Writing a list to a file**
>
> Given `names = ["Kamil", "Ania", "Tomek"]`, write each name on its own line to a file called `names.txt`.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
names = ["Kamil", "Ania", "Tomek"]

with open("names.txt", "w", encoding="utf-8") as file:
    for name in names:
        file.write(name + "\n")

print("names.txt saved")
```
</details>

> 📝 **Exercise 5: Reading and counting lines**
>
> Read the `names.txt` file from the previous exercise and print how many lines (names) it contains, plus each name in uppercase.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
with open("names.txt", "r", encoding="utf-8") as file:
    lines = file.readlines()

print(f"Number of names: {len(lines)}")
for line in lines:
    print(line.strip().upper())
```
</details>

> 📝 **Exercise 6: Appending to a log**
>
> Using `"a"` mode, append a new line `"Event 3"` to the `log.txt` file (from section 7), without erasing its earlier content. Finally, read and print the whole file to check that everything is correct.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
with open("log.txt", "a", encoding="utf-8") as file:
    file.write("Event 3\n")

with open("log.txt", "r", encoding="utf-8") as file:
    print(file.read())
```
</details>

> 📝 **Exercise 7: A custom validation exception**
>
> Write a function `set_password(password)` that raises a `ValueError` with a sensible message if the password is shorter than 8 characters. Otherwise it returns `"Password set"`. Test it inside a `try`/`except` block.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
def set_password(password):
    if len(password) < 8:
        raise ValueError("Password must be at least 8 characters long")
    return "Password set"

try:
    print(set_password("abc"))
except ValueError as error:
    print(f"Error: {error}")

print(set_password("secure123"))
```
</details>

> 🔥 **Exercise 8 (challenge): Importing data from a CSV-like file**
>
> Write a few lines in `name,price,qty` format to a file `products.csv` (e.g. `bread,4.5,20`), and deliberately break one line (e.g. a missing value or a price written as text). Read the file, split each line on commas (`.split(",")`), try to convert the price and quantity to numbers inside a `try`/`except` block, skipping (with a printed warning) any line that can't be parsed correctly, and print the total value of the valid products.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
with open("products.csv", "w", encoding="utf-8") as file:
    file.write("bread,4.5,20\n")
    file.write("milk,3.2,15\n")
    file.write("broken_line,not_a_price,10\n")
    file.write("eggs,12.0,8\n")

total_value = 0

with open("products.csv", "r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        parts = line.strip().split(",")
        try:
            name = parts[0]
            price = float(parts[1])
            qty = int(parts[2])
        except (ValueError, IndexError):
            print(f"Skipping line {line_number} - bad format: {line.strip()}")
            continue

        value = price * qty
        print(f"{name}: ${value:.2f}")
        total_value += value

print(f"Total value of valid products: ${total_value:.2f}")
```

Hint: `enumerate(file, start=1)` gives you a line number counted from 1, which makes it easy to tell the user exactly which line in the file is broken.
</details>

---

### What's next?

Congratulations — that's the end of five modules: from `print()` all the way to files
and error handling. That's a solid foundation to build on next: classes and
object-oriented programming, working with external libraries (e.g. `requests`,
`pandas`), or a real project that ties it all together. Let me know which direction you
want to take next.